# CartPole Q-Learning with Bellman Updates

## CartPole Environment Description

The `CartPole-v1` environment from the Gym library models a cart with a pole hinged to it. The agent interacts with the environment as follows:

- The state consists of four values:
$$
s_t = (x_t, \dot{x}_t, \theta_t, \dot{\theta}_t)
$$
where $x_t$ is the cart's position, $\dot{x}_t$ is its linear velocity, $\theta_t$ is the pole's angle from vertical, and $\dot{\theta}_t$ is its angular velocity.
- Actions: $a_t \in \{0, 1\}$, where 0 pushes the cart left and 1 pushes it right.
- Reward: $r_t = 1$ for every step the system stays within bounds.
- The episode ends when $|x_t| > 4.8$, $|\theta_t| > 24^{\circ}$, or 500 steps have elapsed.

In the code, the continuous state is discretized onto a uniform grid (the `Discretizer` class), so a tabular agent can store Q-values for a finite number of state cells. This lets the classic Q-learning algorithm be applied to the original continuous-state-space problem.

## The Q-function update

In tabular Q-learning, the action-value estimate is updated via the Bellman equation:
$$
Q_{t+1}(s_t, a_t) = (1 - \alpha) Q_t(s_t, a_t) + \alpha \left[r_t + \gamma \max_{a'} Q_t(s_{t+1}, a')\right]
$$
where $\alpha$ is the learning rate and $\gamma$ is the discount factor, controlling how much future rewards contribute.

The action-selection strategy is implemented via an $\varepsilon$-greedy policy:
$$
\pi(a \mid s) =
\begin{cases}
\arg\max_a Q(s, a), & \text{with probability } 1 - \varepsilon, \\n\text{a random action}, & \text{with probability } \varepsilon.
\end{cases}
$$
The `eps_start`, `eps_end`, and `eps_decay` parameters set $\varepsilon$'s decay schedule, letting the agent explore the state space aggressively at first and then focus on exploiting the policy it has found.

In [ ]:
!pip install -U pip
!pip install "gymnasium[classic-control]" pygame

  Using cached pygame-2.6.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (12 kB)
Using cached pygame-2.6.1-cp313-cp313-macosx_11_0_arm64.whl (12.4 MB)


In [13]:
import argparse
import os
import warnings
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Tuple

import gym
import numpy as np
from gym.wrappers import RecordVideo
from tqdm.auto import tqdm

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    message="`np.bool8` is a deprecated alias for `np.bool_`."
)
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="pygame.pkgdata"
)

In [14]:
@dataclass
class QLearningConfig:
    """Configuration bundle that controls training, evaluation, and discretisation behaviour."""
    num_episodes: int = 4000
    max_steps_per_episode: int = 500
    learning_rate: float = 0.1
    discount_factor: float = 0.99
    epsilon_start: float = 1.0
    epsilon_end: float = 0.05
    epsilon_decay_episodes: int = 2000
    seed: int = 42
    # Discretization bins per state dimension: [x, x_dot, theta, theta_dot]
    bins: Tuple[int, int, int, int] = (8, 8, 16, 16)
    model_output_path: str = "cartpole_q_table.npy"

In [ ]:
class Discretizer:
    """Map continuous CartPole observations onto a finite lattice of bins."""

    def __init__(self, bins: Tuple[int, int, int, int]):
        """Pre-compute clipping ranges and bin edges for every state dimension.

        Args:
            bins: Number of bins for (x, x_dot, theta, theta_dot).
        """
        # Reasonable clipping ranges for CartPole-v1
        self.bins = bins
        # cart position
        self.x_range = (-4.8, 4.8)
        # cart velocity (not bounded in env spec) – clip to a sensible range
        self.x_dot_range = (-3.0, 3.0)
        # pole angle (radians) ~ +/- 24 degrees
        self.theta_range = (-0.418, 0.418)
        # pole angular velocity – clip to a sensible range
        self.theta_dot_range = (-3.5, 3.5)

        self.bin_edges = [
            np.linspace(self.x_range[0], self.x_range[1], bins[0] - 1),
            np.linspace(self.x_dot_range[0], self.x_dot_range[1], bins[1] - 1),
            np.linspace(self.theta_range[0], self.theta_range[1], bins[2] - 1),
            np.linspace(self.theta_dot_range[0], self.theta_dot_range[1], bins[3] - 1),
        ]

    def clip_state(self, state: np.ndarray) -> np.ndarray:
        """Clip raw observations to stabilise downstream binning.

        Args:
            state: Continuous observation [x, x_dot, theta, theta_dot].

        Returns:
            Clipped observation with values limited to admissible ranges.
        """
        x, x_dot, theta, theta_dot = state
        x = np.clip(x, *self.x_range)
        x_dot = np.clip(x_dot, *self.x_dot_range)
        theta = np.clip(theta, *self.theta_range)
        theta_dot = np.clip(theta_dot, *self.theta_dot_range)
        return np.array([x, x_dot, theta, theta_dot], dtype=np.float32)

    def discretize(self, state: np.ndarray) -> Tuple[int, int, int, int]:
        """Convert a continuous observation into a tuple of bin indices.

        Args:
            state: Continuous observation from the environment.

        Returns:
            Tuple of per-dimension bin indices after clipping and digitisation.
        """
        clipped = self.clip_state(state)
        idxs = [int(np.digitize(clipped[i], self.bin_edges[i])) for i in range(4)]
        # ensure indices are within [0, bins_i - 1]
        idxs = [min(max(0, idx), self.bins[i] - 1) for i, idx in enumerate(idxs)]
        return tuple(idxs)

    def flat_index(self, indices: Tuple[int, int, int, int]) -> int:
        """Flatten multi-dimensional bin indices into a single integer index.

        Args:
            indices: Tuple of bin indices (i_x, i_xdot, i_theta, i_thetadot).

        Returns:
            Linearised index compatible with the tabular Q-table layout.
        """
        b0, b1, b2, b3 = self.bins
        i0, i1, i2, i3 = indices
        return ((i0 * b1 + i1) * b2 + i2) * b3 + i3

    @property
    def num_states(self) -> int:
        """Total number of discrete states produced by the discretizer.

        Returns:
            Number of unique discrete states across all dimensions.
        """
        return int(np.prod(self.bins))

In [16]:
class QLearningAgent:
    """Tabular epsilon-greedy Q-learning agent for discretised CartPole observations."""

    def __init__(self, env: gym.Env, config: QLearningConfig):
        """Initialise buffers, RNG, and helper structures required for learning.

        Args:
            env: Gym environment exposing the CartPole dynamics.
            config: Hyperparameters and discretisation settings.
        """
        self.env = env
        self.config = config
        self.discretizer = Discretizer(config.bins)
        self.num_actions = env.action_space.n
        self.q_table = np.zeros((self.discretizer.num_states, self.num_actions), dtype=np.float32)
        self.rng = np.random.default_rng(config.seed)

    def _epsilon(self, episode: int) -> float:
        """Return exploration rate for the given episode using linear decay.

        Args:
            episode: Zero-based training episode index.

        Returns:
            Exploration probability epsilon for epsilon-greedy action selection.
        """
        if episode >= self.config.epsilon_decay_episodes:
            return self.config.epsilon_end
        frac = episode / max(1, self.config.epsilon_decay_episodes)
        return self.config.epsilon_start + frac * (self.config.epsilon_end - self.config.epsilon_start)

    def _state_to_index(self, obs: np.ndarray) -> int:
        """Map a continuous observation to its discrete state index.

        Args:
            obs: Continuous observation returned by the environment.

        Returns:
            Integer index into the flattened Q-table.
        """
        idxs = self.discretizer.discretize(obs)
        return self.discretizer.flat_index(idxs)

    def act(self, state_idx: int, epsilon: float) -> int:
        """Choose an action via epsilon-greedy policy with respect to the current Q-table.

        Args:
            state_idx: Discrete index representing the current state.
            epsilon: Exploration probability for the decision.

        Returns:
            Selected action index from the discrete action space.
        """
        if self.rng.random() < epsilon:
            return int(self.rng.integers(self.num_actions))
        return int(np.argmax(self.q_table[state_idx]))

    def bellman_update(self, s_idx: int, a: int, r: float, s_next_idx: int, done: bool) -> None:
        """Apply a single Q-learning Bellman update for state-action pair (s, a).

        Args:
            s_idx: Discrete index of the current state.
            a: Executed action index.
            r: Immediate reward received from the environment.
            s_next_idx: Discrete index of the successor state.
            done: Flag indicating whether the transition terminated the episode.
        """
        best_next = 0.0 if done else float(np.max(self.q_table[s_next_idx]))
        td_target = r + self.config.discount_factor * best_next
        td_error = td_target - float(self.q_table[s_idx, a])
        self.q_table[s_idx, a] += self.config.learning_rate * td_error

    def train(self) -> Tuple[np.ndarray, np.ndarray]:
        """Run Q-learning for the configured number of episodes.

        A tqdm progress bar is emitted so long training runs provide feedback.

        Returns:
            Tuple of arrays (episode_returns, episode_lengths) where the first element
            holds the cumulative reward collected per episode and the second element
            records how many steps each episode lasted.
        """
        episode_returns = np.zeros(self.config.num_episodes, dtype=np.float32)
        episode_lengths = np.zeros(self.config.num_episodes, dtype=np.int32)

        for ep in tqdm(range(self.config.num_episodes), desc="Training episodes", unit="ep"):
            obs, _ = self.env.reset(seed=self.config.seed + ep)
            s_idx = self._state_to_index(obs)
            epsilon = self._epsilon(ep)
            total_reward = 0.0
            done = False

            for t in range(self.config.max_steps_per_episode):
                action = self.act(s_idx, epsilon)
                next_obs, reward, terminated, truncated, _ = self.env.step(action)
                done = terminated or truncated
                s_next_idx = self._state_to_index(next_obs)

                self.bellman_update(s_idx, action, float(reward), s_next_idx, done)

                total_reward += float(reward)
                s_idx = s_next_idx

                if done:
                    episode_returns[ep] = total_reward
                    episode_lengths[ep] = t + 1
                    break

            if not done:
                episode_returns[ep] = total_reward
                episode_lengths[ep] = self.config.max_steps_per_episode

            # Optional: small learning-rate decay could help stability
        return episode_returns, episode_lengths

    def evaluate(self, num_episodes: int = 10) -> Tuple[float, float]:
        """Evaluate a greedy policy and return mean episode return and length.

        Args:
            num_episodes: Number of evaluation rollouts to perform.

        Returns:
            Tuple (mean_return, mean_length) summarising the evaluation rollouts.
        """
        returns = []
        lengths = []
        for ep in range(num_episodes):
            obs, _ = self.env.reset(seed=self.config.seed + 10_000 + ep)
            s_idx = self._state_to_index(obs)
            total_reward = 0.0
            done = False
            for t in range(self.config.max_steps_per_episode):
                action = int(np.argmax(self.q_table[s_idx]))
                next_obs, reward, terminated, truncated, _ = self.env.step(action)
                total_reward += float(reward)
                s_idx = self._state_to_index(next_obs)
                done = terminated or truncated
                if done:
                    returns.append(total_reward)
                    lengths.append(t + 1)
                    break
            if not done:
                returns.append(total_reward)
                lengths.append(self.config.max_steps_per_episode)
        return float(np.mean(returns)), float(np.mean(lengths))

    def save(self, path: str) -> None:
        """Persist the learned Q-table to disk, creating directories if required.

        Args:
            path: Destination path for the `.npy` artefact.
        """
        directory = os.path.dirname(path)
        if directory:
            os.makedirs(directory, exist_ok=True)
        np.save(path, self.q_table)

    def load(self, path: str) -> None:
        """Load a previously saved Q-table after validating its structure.

        Args:
            path: Path to a saved numpy array containing a Q-table.

        Raises:
            ValueError: If the loaded table does not match expected dimensions.
        """
        arr = np.load(path, allow_pickle=False)
        expected_shape = (self.discretizer.num_states, self.num_actions)
        if arr.shape != expected_shape:
            raise ValueError(f"Q-table shape mismatch: got {arr.shape}, expected {expected_shape}")
        self.q_table = arr.astype(np.float32, copy=False)

In [17]:
def _run_greedy_episode(agent: QLearningAgent, env: gym.Env, seed: int) -> Tuple[float, int]:
    """Roll out a single greedy episode for video or evaluation utilities.

    Args:
        agent: Agent providing the greedy policy and discretisation logic.
        env: Environment used for the rollout (may include rendering wrappers).
        seed: Environment seed applied before the episode.

    Returns:
        Tuple containing total reward and number of steps executed.
    """
    obs, _ = env.reset(seed=seed)
    s_idx = agent._state_to_index(obs)
    total_reward = 0.0
    for t in range(agent.config.max_steps_per_episode):
        action = int(np.argmax(agent.q_table[s_idx]))
        next_obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += float(reward)
        s_idx = agent._state_to_index(next_obs)
        if terminated or truncated:
            return total_reward, t + 1
    return total_reward, agent.config.max_steps_per_episode

In [18]:
def train_and_evaluate(args: argparse.Namespace) -> None:
    """Train the agent, report metrics, save artefacts, and optionally record evaluation videos.

    Args:
        args: Parsed CLI configuration namespace.
    """
    env = gym.make("CartPole-v1")

    try:
        config = QLearningConfig(
            num_episodes=args.episodes,
            max_steps_per_episode=args.max_steps,
            learning_rate=args.lr,
            discount_factor=args.gamma,
            epsilon_start=args.eps_start,
            epsilon_end=args.eps_end,
            epsilon_decay_episodes=args.eps_decay_episodes,
            seed=args.seed,
            bins=(args.bins_x, args.bins_xdot, args.bins_theta, args.bins_thetadot),
            model_output_path=args.output,
        )

        agent = QLearningAgent(env, config)
        returns, lengths = agent.train()

        mean_last_100 = float(np.mean(returns[-100:])) if len(returns) >= 100 else float(np.mean(returns))
        eval_return, eval_length = agent.evaluate(num_episodes=args.eval_episodes)

        agent.save(config.model_output_path)

        print(f"Training complete. Mean return (last 100): {mean_last_100:.2f}")
        print(f"Evaluation over {args.eval_episodes} episodes -> mean return: {eval_return:.2f}, mean length: {eval_length:.1f}")
        print(f"Q-table saved to: {os.path.abspath(config.model_output_path)}")

        # Optional: record evaluation videos
        if args.video_dir:
            base_dir = Path(args.video_dir)
            base_dir.mkdir(parents=True, exist_ok=True)
            name_prefix = f"qlearn_eval_after_{config.num_episodes}"
            timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
            run_dir = base_dir / f"{name_prefix}_{timestamp}"
            run_dir.mkdir(parents=True, exist_ok=True)

            video_env = gym.make("CartPole-v1", render_mode="rgb_array")
            video_env = RecordVideo(
                video_env,
                video_folder=str(run_dir),
                episode_trigger=lambda ep_id: True,
                name_prefix=name_prefix,
            )
            try:
                print(
                    f"Recording {args.video_episodes} evaluation episode(s) to: {run_dir.resolve()}"
                )
                rec_returns = []
                rec_lengths = []
                for ep in range(args.video_episodes):
                    ret, ln = _run_greedy_episode(agent, video_env, seed=config.seed + 20_000 + ep)
                    rec_returns.append(ret)
                    rec_lengths.append(ln)
                if rec_returns:
                    print(
                        f"Recorded episodes -> mean return: {float(np.mean(rec_returns)):.2f}, "
                        f"mean length: {float(np.mean(rec_lengths)):.1f}"
                    )
            finally:
                video_env.close()
    finally:
        env.close()

In [19]:
def make_args(
    episodes: int = 4000,
    max_steps: int = 500,
    lr: float = 0.1,
    gamma: float = 0.99,
    eps_start: float = 1.0,
    eps_end: float = 0.05,
    eps_decay_episodes: int = 2000,
    seed: int = 42,
    bins_x: int = 8,
    bins_xdot: int = 8,
    bins_theta: int = 16,
    bins_thetadot: int = 16,
    eval_episodes: int = 10,
    output: str = "cartpole_q_table.npy",
    video_dir: str = "videos/cartpole",
    video_episodes: int = 1,
) -> argparse.Namespace:
    """Utility helper to create an argparse-like namespace for notebook experiments."""
    return argparse.Namespace(
        episodes=episodes,
        max_steps=max_steps,
        lr=lr,
        gamma=gamma,
        eps_start=eps_start,
        eps_end=eps_end,
        eps_decay_episodes=eps_decay_episodes,
        seed=seed,
        bins_x=bins_x,
        bins_xdot=bins_xdot,
        bins_theta=bins_theta,
        bins_thetadot=bins_thetadot,
        eval_episodes=eval_episodes,
        output=output,
        video_dir=video_dir,
        video_episodes=video_episodes,
    )

In [ ]:
args = make_args(
    episodes=4000,
    eval_episodes=10,
    video_episodes=3,
    video_dir="videos/cartpole"
)
train_and_evaluate(args)

Training episodes:   0%|          | 0/4000 [00:00<?, ?ep/s]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Training complete. Mean return (last 100): 328.14
Evaluation over 10 episodes -> mean return: 360.80, mean length: 360.8
Q-table saved to: /Users/denissamatov/ML/RL/RL_course/code/cartpole_q_table.npy
Recording 3 evaluation episode(s) to: /Users/denissamatov/ML/RL/RL_course/code/videos/cartpole


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/gym/wrappers/record_video.py:75: UserWarning: WARN: Overwriting existing videos at /Users/denissamatov/ML/RL/RL_course/code/videos/cartpole folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bo

MoviePy - Building video /Users/denissamatov/ML/RL/RL_course/code/videos/cartpole/qlearn_eval_after_4000-episode-0.mp4.
MoviePy - Writing video /Users/denissamatov/ML/RL/RL_course/code/videos/cartpole/qlearn_eval_after_4000-episode-0.mp4



MoviePy - Done !
MoviePy - video ready /Users/denissamatov/ML/RL/RL_course/code/videos/cartpole/qlearn_eval_after_4000-episode-0.mp4


MoviePy - Building video /Users/denissamatov/ML/RL/RL_course/code/videos/cartpole/qlearn_eval_after_4000-episode-1.mp4.
MoviePy - Writing video /Users/denissamatov/ML/RL/RL_course/code/videos/cartpole/qlearn_eval_after_4000-episode-1.mp4



MoviePy - Done !
MoviePy - video ready /Users/denissamatov/ML/RL/RL_course/code/videos/cartpole/qlearn_eval_after_4000-episode-1.mp4
MoviePy - Building video /Users/denissamatov/ML/RL/RL_course/code/videos/cartpole/qlearn_eval_after_4000-episode-2.mp4.
MoviePy - Writing video /Users/denissamatov/ML/RL/RL_course/code/videos/cartpole/qlearn_eval_after_4000-episode-2.mp4



MoviePy - Done !
MoviePy - video ready /Users/denissamatov/ML/RL/RL_course/code/videos/cartpole/qlearn_eval_after_4000-episode-2.mp4
Recorded episodes -> mean return: 331.00, mean length: 331.0
